# Task 1 - Manual GPT training (Aswin)
Run after `preprocessing.ipynb`, in the same uploaded repository folder. The model uses manual Q/K/V projections, scaled dot-product attention, head reshaping, and a lower-triangular causal mask. It does not use `nn.Transformer`, `nn.MultiheadAttention`, or any pretrained component. The full config trains 12 epochs.

In [ ]:
from pathlib import Path
import importlib.util, os, subprocess, sys

packages = {'yaml': 'pyyaml', 'psutil': 'psutil', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'tqdm': 'tqdm'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

def locate_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'task1_llm/member_aswin/configs/full.yaml').exists(): return candidate.resolve()
    content = Path('/content')
    matches = list(content.rglob('task1_llm/member_aswin/configs/full.yaml')) if content.exists() else []
    if not matches:
        try:
            from google.colab import drive
            print('Repository not found in /content; mounting Google Drive ...')
            drive.mount('/content/drive')
            drive_root = Path('/content/drive/MyDrive')
            matches = list(drive_root.rglob('task1_llm/member_aswin/configs/full.yaml'))
        except ImportError:
            pass
    if not matches: raise FileNotFoundError('Upload the complete folder containing task1_llm to /content or My Drive.')
    return matches[0].resolve().parents[3]

REPO_ROOT = locate_repo_root(); os.chdir(REPO_ROOT)
CONFIG = REPO_ROOT / 'task1_llm/member_aswin/configs/full.yaml'
SCRIPT = REPO_ROOT / 'task1_llm/member_aswin/src/train.py'
import torch, yaml
RUN_ID = yaml.safe_load(CONFIG.read_text())['run_id']
processed = REPO_ROOT / 'task1_llm/member_aswin/data_processed' / RUN_ID
if not (processed / 'train.pt').exists() or not (processed / 'val.pt').exists():
    raise FileNotFoundError('Processed data is missing. Run preprocessing.ipynb first in the same uploaded folder.')
if not torch.cuda.is_available():
    raise RuntimeError('GPU not detected. In Colab choose Runtime > Change runtime type > GPU, then restart.')
print('Repository root:', REPO_ROOT)
print('GPU:', torch.cuda.get_device_name(0))
print('Run ID:', RUN_ID)
subprocess.run([sys.executable, str(SCRIPT), '--config', str(CONFIG)], cwd=REPO_ROOT, check=True)

In [ ]:
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv(REPO_ROOT / 'task1_llm/member_aswin/metrics_report.csv'))
display(Image(filename=str(REPO_ROOT / 'task1_llm/member_aswin/outputs/plots' / f'{RUN_ID}_loss_curves.png')))